<h1> Libraries </h1>

In [ ]:
from __future__ import annotations

import os
import json
from typing import Optional
import pandas as pd
import numpy as np
from pathlib

from data import Verdict
from utils import get_country, companies_from_dataframe, parse_nested, company_id, distance_km, geo_penalty, naics_filter_mode, _naics_codes
from embedding_client import LocalEmbeddingClient, build_embedding_dataset
from llm_client import LocalLLMClient, REGION_ALIASES

In [3]:
df = pd.read_json("data/companies.jsonl", lines=True)

<h1> Data Visualization </h1>

In [4]:
null_naics = df["primary_naics"].isna().sum()
print(f"Null primary_naics values: {null_naics}")


Null primary_naics values: 0


In [5]:
null_naics = df["secondary_naics"].isna().sum()
print(f"Null secondary_naics values: {null_naics}")

Null secondary_naics values: 466


In [6]:
completeness = pd.DataFrame({
    "field": df.columns,
    "missing": df.isna().sum().values,
    "missing_pct": (df.isna().mean() * 100).round(2).values,
    "present": df.notna().sum().values,
    "present_pct": (df.notna().mean() * 100).round(2).values,
})

completeness = completeness.sort_values(
    "missing_pct",
    ascending=False
).reset_index(drop=True)

print(completeness.to_string(index=False))

           field  missing  missing_pct  present  present_pct
 secondary_naics      466        97.69       11         2.31
  employee_count      188        39.41      289        60.59
    year_founded      131        27.46      346        72.54
         revenue       93        19.50      384        80.50
         website       38         7.97      439        92.03
operational_name        2         0.42      475        99.58
         address        0         0.00      477       100.00
     description        0         0.00      477       100.00
   primary_naics        0         0.00      477       100.00
  business_model        0         0.00      477       100.00
  target_markets        0         0.00      477       100.00
  core_offerings        0         0.00      477       100.00
       is_public        0         0.00      477       100.00


In [7]:
# ------------------------------------------------------------
# 1. COUNTRY DISTRIBUTION
# ------------------------------------------------------------
country_distribution = (
    df["address"]
    .apply(get_country)
    .value_counts(dropna=False)
    .rename_axis("country")
    .reset_index(name="count")
)

country_distribution["pct"] = (
    country_distribution["count"] / len(df) * 100
).round(2)

print("\nCOUNTRY DISTRIBUTION")
print(country_distribution.to_string(index=False))


COUNTRY DISTRIBUTION
                  country  count   pct
            United States     86 18.03
              Switzerland     43  9.01
                   France     40  8.39
                    China     35  7.34
                   Sweden     31  6.50
           United Kingdom     27  5.66
                   Norway     27  5.66
                  Romania     26  5.45
                  Denmark     26  5.45
                    Spain     19  3.98
                  Finland     13  2.73
                  Germany     12  2.52
                    India     11  2.31
              Netherlands      8  1.68
                Australia      8  1.68
                   Canada      8  1.68
                    Italy      5  1.05
                  Ireland      4  0.84
                Singapore      4  0.84
                  Iceland      4  0.84
       Korea, Republic of      4  0.84
                  Belgium      3  0.63
                   Poland      3  0.63
               Luxembourg      3  0.63
   

In [8]:
# ------------------------------------------------------------
# 2. PRIMARY NAICS DISTRIBUTION
# ------------------------------------------------------------

def get_naics(value):
    value = parse_nested(value)

    if not value:
        return None

    return value.get("code")


def get_naics_label(value):
    value = parse_nested(value)

    if not value:
        return None

    return value.get("label")


naics = pd.DataFrame({
    "code": df["primary_naics"].apply(get_naics),
    "label": df["primary_naics"].apply(get_naics_label),
})

naics_distribution = (
    naics
    .value_counts(["code", "label"], dropna=False)
    .reset_index(name="count")
)

naics_distribution["pct"] = (
    naics_distribution["count"] / len(df) * 100
).round(2)

print("\nPRIMARY NAICS DISTRIBUTION")
print(naics_distribution.to_string(index=False))



PRIMARY NAICS DISTRIBUTION
  code                                                                                                              label  count   pct
513210                                                                                                Software Publishers     67 14.05
333611                                                              Turbine and Turbine Generator Set Units Manufacturing     44  9.22
541512                                                                                   Computer Systems Design Services     30  6.29
325412                                                                           Pharmaceutical Preparation Manufacturing     25  5.24
335910                                                                                              Battery Manufacturing     24  5.03
541330                                                                                               Engineering Services     22  4.61
325180                     

In [9]:
# ------------------------------------------------------------
# 3. EMPLOYEE COUNT DISTRIBUTION
# ------------------------------------------------------------

employee_count = pd.to_numeric(
    df["employee_count"],
    errors="coerce"
)

employee_distribution = pd.Series({
    "count": employee_count.notna().sum(),
    "missing": employee_count.isna().sum(),
    "min": employee_count.min(),
    "25%": employee_count.quantile(0.25),
    "median": employee_count.median(),
    "75%": employee_count.quantile(0.75),
    "max": employee_count.max(),
    "mean": employee_count.mean(),
})

print("\nEMPLOYEE COUNT DISTRIBUTION")
print(employee_distribution)


# Optional: useful buckets for understanding the dataset
employee_buckets = pd.cut(
    employee_count,
    bins=[
        -float("inf"),
        10,
        50,
        100,
        500,
        1000,
        5000,
        10000,
        float("inf"),
    ],
    labels=[
        "<=10",
        "11-50",
        "51-100",
        "101-500",
        "501-1K",
        "1K-5K",
        "5K-10K",
        "10K+",
    ],
)

employee_bucket_distribution = (
    employee_buckets
    .value_counts(sort=False, dropna=False)
    .rename_axis("employee_range")
    .reset_index(name="count")
)

employee_bucket_distribution["pct"] = (
    employee_bucket_distribution["count"] / len(df) * 100
).round(2)

print("\nEMPLOYEE COUNT BUCKETS")
print(employee_bucket_distribution.to_string(index=False))


EMPLOYEE COUNT DISTRIBUTION
count      2.890000e+02
missing    1.880000e+02
min        1.000000e+00
25%        5.000000e+00
median     3.000000e+01
75%        2.603000e+03
max        2.100000e+06
mean       2.509067e+04
dtype: float64

EMPLOYEE COUNT BUCKETS
employee_range  count   pct
          <=10    108 22.64
         11-50     48 10.06
        51-100     12  2.52
       101-500     28  5.87
        501-1K     10  2.10
         1K-5K     17  3.56
        5K-10K     10  2.10
          10K+     56 11.74
           NaN    188 39.41


In [10]:
# ------------------------------------------------------------
# 4. REVENUE DISTRIBUTION
# ------------------------------------------------------------

revenue = pd.to_numeric(
    df["revenue"],
    errors="coerce"
)

revenue_distribution = pd.Series({
    "count": revenue.notna().sum(),
    "missing": revenue.isna().sum(),
    "min": revenue.min(),
    "25%": revenue.quantile(0.25),
    "median": revenue.median(),
    "75%": revenue.quantile(0.75),
    "max": revenue.max(),
    "mean": revenue.mean(),
})

print("\nREVENUE DISTRIBUTION")
print(revenue_distribution)


# Optional: revenue buckets
revenue_buckets = pd.cut(
    revenue,
    bins=[
        -float("inf"),
        1_000_000,
        10_000_000,
        100_000_000,
        1_000_000_000,
        10_000_000_000,
        100_000_000_000,
        float("inf"),
    ],
    labels=[
        "<$1M",
        "$1M-$10M",
        "$10M-$100M",
        "$100M-$1B",
        "$1B-$10B",
        "$10B-$100B",
        "$100B+",
    ],
)

revenue_bucket_distribution = (
    revenue_buckets
    .value_counts(sort=False, dropna=False)
    .rename_axis("revenue_range")
    .reset_index(name="count")
)

revenue_bucket_distribution["pct"] = (
    revenue_bucket_distribution["count"] / len(df) * 100
).round(2)

print("\nREVENUE BUCKETS")
print(revenue_bucket_distribution.to_string(index=False))


REVENUE DISTRIBUTION
count      3.840000e+02
missing    9.300000e+01
min        7.524000e+03
25%        2.483684e+06
median     2.018448e+07
75%        1.592155e+09
max        2.500389e+12
mean       1.146714e+10
dtype: float64

REVENUE BUCKETS
revenue_range  count   pct
         <$1M     71 14.88
     $1M-$10M     84 17.61
   $10M-$100M     79 16.56
    $100M-$1B     40  8.39
     $1B-$10B     46  9.64
   $10B-$100B     60 12.58
       $100B+      4  0.84
          NaN     93 19.50


<h1> Local Embedding Client </h1>

In [11]:
embedding_client = LocalEmbeddingClient(
    model_name="BAAI/bge-small-en-v1.5",
    batch_size=64,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7532.14it/s]


In [12]:
companies = companies_from_dataframe(df)

In [13]:
build_embedding_dataset(
    companies,
    embedding_client,
    output_dir="data/embeddings",
)

Companies: 477
Already embedded: 477
Need embedding: 0
No new or changed companies. Nothing to embed.


<h1> LLM CLient </h1>

In [14]:
llm_client = LocalLLMClient()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:14<00:00, 24.16it/s]


<h1> Stage 1: Structured Filtering </h1>

In [15]:
def structured_filter(companies: list, filters: dict) -> list:
    """Returns [(company, low_confidence), ...] for companies that pass."""
    out = []
    for c in companies:
        low_conf = False
        ok = True
        if ok and "country" in filters:
            country = (c.address or {}).get("country")
            if not country:
                low_conf = True
            else:
                targets = filters["country"]
                if isinstance(targets, str):
                    targets = [targets]
                if not any(
                    str(target).strip().lower() == country.strip().lower()
                    for target in targets
                ):
                    ok = False
        
        if ok and filters.get("employee_count_min") is not None:
            if c.employee_count is None:
                low_conf = True
            elif c.employee_count < filters["employee_count_min"]:
                ok = False

        if ok and filters.get("employee_count_max") is not None:
            if c.employee_count is None:
                low_conf = True
            elif c.employee_count > filters["employee_count_max"]:
                ok = False

        if ok and filters.get("revenue_min") is not None:
            if c.revenue is None:
                low_conf = True
            elif c.revenue < filters["revenue_min"]:
                ok = False

        if ok and filters.get("revenue_max") is not None:
            if c.revenue is None:
                low_conf = True
            elif c.revenue > filters["revenue_max"]:
                ok = False

        if ok and filters.get("year_founded_min") is not None:
            if c.year_founded is None:
                low_conf = True
            elif c.year_founded < filters["year_founded_min"]:
                ok = False

        if ok and filters.get("year_founded_max") is not None:
            if c.year_founded is None:
                low_conf = True
            elif c.year_founded > filters["year_founded_max"]:
                ok = False

        if ok and filters.get("is_public") is True:
            if c.is_public is None:
                mentions_public = any(
                    kw in (c.description or "").lower()
                    for kw in ["publicly traded", "listed on", "nyse", "nasdaq", "stock exchange"]
                )
                if mentions_public:
                    low_conf = True
                else:
                    ok = False
            elif c.is_public is not True:
                ok = False

        # if ok and filters.get("naics_prefixes"):
        #     codes = _naics_codes(c)
        #     if not codes:
        #         warnings.warn(
        #             f"No NAICS code for {c.operational_name!r} "
        #             f"(raw: {c.primary_naics!r}) -- check normalize_naics.",
        #             stacklevel=2,
        #         )
        #         ok = False
        #     else:
        #         matches = [
        #             p for code in codes for p in filters["naics_prefixes"]
        #             if code.startswith(p)
        #         ]
        #         strong_matches = [p for p in matches if len(p) >= 1]
        #         if not strong_matches:
        #             ok = False

        if ok:
            out.append((c, low_conf))
    return out


<h1> Stage 2: Embedding Retrieval </h1>

In [16]:
def load_embedding_index(output_dir: str = "data/embeddings") -> dict:
    """company_id -> np.ndarray, loaded from the .npz written by build_embedding_dataset."""
    vectors_path = os.path.join(output_dir, "embeddings.npz")
    if not os.path.exists(vectors_path):
        return {}
    loaded = np.load(vectors_path, allow_pickle=False)
    return {
        str(cid): vec
        for cid, vec in zip(loaded["company_ids"], loaded["embeddings"])
    }


def embedding_retrieve(
    query: str,
    plan: dict,
    candidates: list,
    embedding_index: dict,
    embedding_client,
    top_n: int = 50,
) -> list:
    """Returns [(company, score), ...] sorted descending, length <= top_n."""
    if not candidates:
        return []

    query_texts = [query, plan.get("hypothetical_profile", "")] + plan.get("expansion_terms", [])[:8]
    query_texts = [t for t in query_texts if t]
    query_vecs = embedding_client.embed(query_texts)  # already normalized

    scored = []
    to_embed_live = []  # (index_in_scored_placeholder, company) needing on-the-fly embedding
    for c in candidates:
        cid = company_id({"website": c.website, "operational_name": c.operational_name})
        vec = embedding_index.get(cid)
        if vec is None:
            to_embed_live.append(c)
            continue
        sims = query_vecs @ vec
        score = 0.6 * sims.max() + 0.4 * sims.mean()
        scored.append((c, float(score)))

    if to_embed_live:
        # Cache miss (new/unembedded company) -- embed on the fly rather than
        # silently excluding it. Slower, but correctness > speed for the
        # (hopefully small) uncached slice.
        live_vecs = embedding_client.embed([c.composite_text() for c in to_embed_live])
        for c, vec in zip(to_embed_live, live_vecs):
            sims = query_vecs @ vec
            score = 0.6 * sims.max() + 0.4 * sims.mean()
            scored.append((c, float(score)))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


<h1> Stage 3 + 4: Batched Judging, Fusion & Ranking </h1>

In [17]:
# ---------------------------------------------------------------------------
# Stage 3 orchestration: batching + self-consistency on borderline cases
# ---------------------------------------------------------------------------

def _cid(c) -> str:
    return company_id({"website": c.website, "operational_name": c.operational_name})


def batched_llm_judge(
    query: str,
    plan: dict,
    candidates: list,
    llm_client,
    batch_size: int = 12,
    borderline_band: tuple = (0.4, 0.6),
    self_consistency_runs: int = 1,
) -> dict:
    by_id = {_cid(c): c for c, _ in candidates}
    companies = [c for c, _ in candidates]
    verdicts = {}

    for i in range(0, len(companies), batch_size):
        batch = companies[i:i + batch_size]
        for r in llm_client.judge_batch(query, plan, batch):
            verdicts[r["company_id"]] = r

    borderline_ids = [
        cid for cid, v in verdicts.items()
        if borderline_band[0] <= v["confidence"] <= borderline_band[1]
    ]
    print("borderline", len(borderline_ids))
    for cid in borderline_ids:
        c = by_id[cid]
        votes = []
        for _ in range(self_consistency_runs):
            run = llm_client.judge_batch(query, plan, [c], temperature=0.7)
            votes.append(run[0])
        match_votes = [v["match"] for v in votes]
        majority_match = sum(match_votes) > len(match_votes) / 2
        avg_confidence = sum(v["confidence"] for v in votes) / len(votes)
        agreement = sum(1 for v in match_votes if v == majority_match)
        verdicts[cid] = {
            "company_id": cid,
            "match": majority_match,
            "confidence": round(avg_confidence, 2),
            "reason": f"self-consistency vote ({agreement}/{len(votes)} runs agreed): " + votes[0]["reason"],
        }

    return verdicts


# ---------------------------------------------------------------------------
# Stage 4: fusion & ranking
# ---------------------------------------------------------------------------

def fuse_and_rank(
    query_type: str,
    structured_pass: list,
    embedding_scores: dict,
    llm_verdicts: dict,
    geo_distances=None,
) -> list:
    geo_distances = geo_distances or {}

    results = []

    for c, low_conf in structured_pass:
        cid = _cid(c)

        e = embedding_scores.get(cid)
        v = llm_verdicts.get(cid)

        # ---------------------------------------------------------------
        # Structured-only query
        # ---------------------------------------------------------------
        if query_type == "structured" and e is None and v is None:
            score = 1.0 - (0.15 if low_conf else 0.0)

            # Geographic penalty
            distance = geo_distances.get(cid)
            score -= geo_penalty(distance)
            score = max(0.0, score)

            results.append(
                Verdict(
                    c,
                    score,
                    True,
                    "matched all structured filters",
                    "filter",
                )
            )
            continue

        # ---------------------------------------------------------------
        # Semantic queries: must have been retrieved by embeddings
        # ---------------------------------------------------------------
        if query_type != "structured" and e is None:
            continue

        # ---------------------------------------------------------------
        # LLM rejection
        # ---------------------------------------------------------------
        if v is not None and not v["match"]:
            continue

        # ---------------------------------------------------------------
        # Base score
        # ---------------------------------------------------------------
        w_struct, w_embed, w_llm = 0.2, 0.3, 0.5

        parts = []
        weights = []

        parts.append(
            1.0 - (0.15 if low_conf else 0.0)
        )
        weights.append(w_struct)

        if e is not None:
            parts.append(max(0.0, min(1.0, e)))
            weights.append(w_embed)

        if v is not None:
            parts.append(v["confidence"])
            weights.append(w_llm)

        score = sum(
            p * w
            for p, w in zip(parts, weights)
        ) / sum(weights)

        # ---------------------------------------------------------------
        # Geographic penalty
        # ---------------------------------------------------------------
        distance = geo_distances.get(cid)
        score -= geo_penalty(distance)
        score = max(0.0, score)

        # ---------------------------------------------------------------
        # Result metadata
        # ---------------------------------------------------------------
        reason = (
            v["reason"]
            if v
            else (
                "passed structured filters, ranked by semantic similarity"
                if e is not None
                else "matched structured filters"
            )
        )

        stage = (
            "llm"
            if v is not None
            else (
                "embedding"
                if e is not None
                else "filter"
            )
        )

        results.append(
            Verdict(
                c,
                score,
                True,
                reason,
                stage,
            )
        )

    results.sort(
        key=lambda r: r.score,
        reverse=True,
    )

    return results


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------
SEARCH_RADII_KM = [25, 50, 100, 200, 400]
MIN_GEO_RESULTS = 20

# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------

SEARCH_RADII_KM = [25, 50, 100, 200, 400]
MIN_GEO_RESULTS = 20


class QualificationSystem:
    def __init__(
        self,
        llm_client,
        embedding_client,
        embedding_index: Optional[dict] = None,
    ):
        self.llm_client = llm_client
        self.embedding_client = embedding_client
        self.embedding_index = embedding_index or {}
        self._query_plan_cache = {}

    def _get_plan(self, query: str) -> dict:
        key = query.strip().lower()

        if key not in self._query_plan_cache:
            self._query_plan_cache[key] = (
                self.llm_client.understand_query(query)
            )

        return self._query_plan_cache[key]

    def _geo_search(
        self,
        companies: list,
        plan: dict,
        min_results: int = MIN_GEO_RESULTS,
    ) -> list:
        """
        Search companies around the geographic location specified in the plan.

        Returns:
            [(company, distance_km), ...]

        The search starts with the smallest radius and expands until
        min_results are found, or until the largest radius is reached.
        """

        filters = plan.get("structured_filters", {})

        latitude = filters.get("latitude")
        longitude = filters.get("longitude")

        # No geographic constraint.
        if latitude is None or longitude is None:
            return [(company, None) for company in companies]

        last_results = []

        for radius_km in SEARCH_RADII_KM:
            results = []

            for company in companies:
                address = company.address or {}

                lat = address.get("latitude")
                lon = address.get("longitude")

                if lat is None or lon is None:
                    continue

                distance = distance_km(
                    float(latitude),
                    float(longitude),
                    float(lat),
                    float(lon),
                )

                if distance <= radius_km:
                    results.append((company, distance))

            if len(results) >= min_results:
                return results

            last_results = results

        return last_results

    def run(
            self,
            query: str,
            companies: list,
            top_n_embed: int = 40,
        ) -> list:

            # ------------------------------------------------------------------
            # 1. Understand query
            # ------------------------------------------------------------------

            plan = self._get_plan(query)
            qtype = plan["query_type"]

            # Defensive net
            if qtype in ("ecosystem", "semantic"):
                plan["structured_filters"].pop("naics_prefixes", None)

            # ------------------------------------------------------------------
            # 2. Hard structured filters
            # ------------------------------------------------------------------

            mode = naics_filter_mode(companies, plan["structured_filters"])
            print(mode)

            structured_pass = structured_filter(
                companies,
                plan["structured_filters"],
            )

            print("Structured pass:", structured_pass)

            candidates = [c for c, _ in structured_pass]

            # ------------------------------------------------------------------
            # 3. Geographic search
            # ------------------------------------------------------------------

            geo_candidates = self._geo_search(
                candidates,
                plan,
            )

            print("Geo candidates:", geo_candidates)

            candidates = [
                company
                for company, _distance in geo_candidates
            ]

            geo_distances = {
                _cid(company): distance
                for company, distance in geo_candidates
            }

            # ------------------------------------------------------------------
            # 4. Structured-only query
            # ------------------------------------------------------------------

            if qtype == "structured":
                return fuse_and_rank(
                    qtype,
                    structured_pass,
                    {},
                    {},
                    geo_distances=geo_distances,
                )

            # ------------------------------------------------------------------
            # 5. Semantic retrieval
            # ------------------------------------------------------------------

            embed_ranked = embedding_retrieve(
                query,
                plan,
                candidates,
                self.embedding_index,
                self.embedding_client,
                top_n=top_n_embed,
            )

            print("Embed_ranked:", embed_ranked)

            embedding_scores = {
                _cid(c): s
                for c, s in embed_ranked
            }

            # ------------------------------------------------------------------
            # 6. Hybrid shortcut
            # ------------------------------------------------------------------

            if qtype == "hybrid" and len(embed_ranked) <= 15:
                return fuse_and_rank(
                    qtype,
                    structured_pass,
                    embedding_scores,
                    {},
                    geo_distances=geo_distances,
                )



            # ------------------------------------------------------------------
            # 7. LLM judging
            # ------------------------------------------------------------------

            llm_verdicts = batched_llm_judge(
                query,
                plan,
                embed_ranked,
                self.llm_client,
            )

            print("LLM Verdicts:", llm_verdicts)

            # ------------------------------------------------------------------
            # 8. Final ranking
            # ------------------------------------------------------------------

            return fuse_and_rank(
                qtype,
                structured_pass,
                embedding_scores,
                llm_verdicts,
                geo_distances=geo_distances,
            )

<h1> Try it out </h1>

In [18]:
embedding_index = load_embedding_index("data/embeddings")
print(f"Loaded {len(embedding_index)} cached embeddings")
system = QualificationSystem(llm_client, embedding_client, embedding_index=embedding_index)

Loaded 456 cached embeddings


In [ ]:
plan = llm_client.understand_query("B2B SaaS companies providing HR solutions in Europe")
print(json.dumps(plan, indent=2))

{
  "query_type": "structured",
  "structured_filters": {
    "country": [
      "Austria",
      "Belgium",
      "Switzerland",
      "Germany",
      "Denmark",
      "Spain",
      "Finland",
      "France",
      "United Kingdom",
      "Greece",
      "Croatia",
      "Ireland",
      "Iceland",
      "Italy",
      "Latvia",
      "Luxembourg",
      "Netherlands",
      "Norway",
      "Poland",
      "Portugal",
      "Romania",
      "Sweden",
      "Ukraine"
    ],
    "naics_prefixes": [
      "33"
    ]
  },
  "expansion_terms": [
    "wind turbine",
    "solar panel manufacturer",
    "renewable energy",
    "green technology",
    "energy solutions"
  ],
  "hypothetical_profile": "We are a leading European manufacturer of high-efficiency wind turbines, specializing in the design and production of state-of-the-art renewable energy solutions for sustainable power generation."
}


In [20]:
import gc
gc.collect()

41

In [ ]:
example_queries = [
    #"Turbine manufacturers in Europe.",
    #"Public software companies with more than 1,000 employees.",
    # "Food and beverage manufacturers in France",
    #"Companies that could supply packaging materials for a direct-to-consumer cosmetics brand",
    # "Construction companies in the United States with revenue over $50 million",
    # "Pharmaceutical companies in Switzerland",
    #"B2B SaaS companies providing HR solutions in Europe",
    # "Clean energy startups founded after 2018 with fewer than 200 employees",
    # "Fast-growing fintech companies competing with traditional banks in Europe.",
    #"E-commerce companies using Shopify or similar platforms",
    #"Renewable energy equipment manufacturers in Scandinavia",
    #"Companies that manufacture or supply critical components for electric vehicle battery production",
]

for q in example_queries:
    print(f"\n=== {q!r} ===")
    results = system.run(q, companies)
    print(f"  ({len(results)} total matches)")
    for r in results:
        print(f"  [{r.stage_reached:9s}] {r.score:.2f}  {r.company.operational_name:30s} - {r.reason}")



=== 'Turbine manufacturers in Europe.' ===
hard
Structured pass: [(Company(operational_name='Rompetrol', website='rompetrol.ro', year_founded=1979, address={'latitude': 44.4775537, 'longitude': 26.0713416, 'region_name': 'Bucharest', 'town': 'Bucharest', 'country': 'Romania'}, employee_count=None, revenue=13498241905.0, primary_naics={'code': '324110', 'label': 'Petroleum Refineries'}, secondary_naics=None, description='Rompetrol is a Romanian company specialized in petroleum refining, petrochemical operations, and the distribution of fuel products. The company operates as a subsidiary of KMG International and manages integrated refineries in Romania, Moldova, Bulgaria, and Georgia, serving the automotive, industrial, and energy sectors. Rompetrol also provides industrial products, wholesale fuel supply, and e-Mobility services, and is certified for quality, health, safety, and environmental management.', business_model=['Wholesale', 'Manufacturing', 'Business-to-Business', 'Retail', 